In [ ]:
!nvidia-smi

In [ ]:
%cd ../

In [1]:
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '2'
os.environ["HF_HOME"] = os.environ["MY_HF_HOME"]

In [ ]:
from typing import List, Union

import torch
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer


class Contriever:
    def __init__(self, model_name='facebook/contriever'):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name, device_map="auto")
        self.batch_size = 128
        self.corpus: List[str] = []
        self.embedding: torch.Tensor = None
    
    @torch.no_grad()
    def _embed(self, input_str):
        encoding = self.tokenizer(input_str, return_tensors='pt', padding=True, truncation=True).to(self.model.device)
        input_ids = encoding['input_ids']
        attention_mask = encoding['attention_mask']
        # input_ids = input_ids.to(device)
        # attention_mask = attention_mask.to(device)

        outputs = self.model(input_ids, attention_mask=attention_mask)
        # embeddings = mean_pooling(outputs[0], attention_mask)
        embeddings = outputs.last_hidden_state
        embeddings = embeddings.masked_fill(~attention_mask[..., None].bool(), 0.)
        embeddings = embeddings.sum(dim=1) / attention_mask.sum(dim=1)[..., None]
        embeddings = embeddings.T.divide(torch.linalg.norm(embeddings, dim=1)).T
        # print(embeddings.shape)
        return embeddings

    def encode_corpus(self, corpus: List[str]):
        self.corpus = corpus
        embeddings = []
        for i in tqdm(range(0, len(corpus), self.batch_size), total=len(corpus)//self.batch_size, desc="Embedding Corpus"):
            embeddings.append(self._embed(corpus[i:i+self.batch_size]))
        self.embedding = torch.cat(embeddings, dim=0)

    def batch_search(self, queries: Union[str, List[str]], top_k=5):
        if isinstance(queries, str):
            queries = [queries]

        query_embedding = []
        for i in tqdm(range(0, len(queries), self.batch_size), total=len(queries)//self.batch_size, desc="Embedding Query"):
            query_embedding.append(self._embed(queries[i:i+self.batch_size]))
        query_embedding = torch.cat(query_embedding, dim=0)

        scores = torch.matmul(query_embedding, self.embedding.T)

        top_k_idx = torch.topk(scores, top_k, dim=1).indices
        docs_to_return = []
        for top_k_idx_row in top_k_idx:
            top_k_docs = [self.corpus[i] for i in top_k_idx_row]
            docs_to_return.append(top_k_docs)
        if len(docs_to_return) == 1:
            docs_to_return = docs_to_return[0]
        return docs_to_return


## MUSIQUE

In [5]:
# musique

import json

with open("reproduce/dataset/musique_corpus.json") as f:
    musique_corpus = json.load(f)
musique_corpus = [item["title"] + '\n' + item["text"] for item in musique_corpus]

with open("reproduce/dataset/musique.json") as f:
    musique_data = json.load(f)

all_queries = [sample['question'] for sample in musique_data]

In [9]:
musique_contriever = Contriever()

In [10]:
musique_contriever.encode_corpus(musique_corpus)

Embedding Corpus: 92it [00:50,  1.82it/s]                        


In [11]:
musique_top_results = musique_contriever.batch_search(all_queries, top_k=100)

Embedding Query: 8it [00:00, 15.88it/s]                       


In [12]:
from collections import defaultdict

k_list = [ 2, 5, 10,20, 50, 100]
total = defaultdict(float)
for sample, retrieved_passages in zip(musique_data, musique_top_results):
    gold_passages = [item for item in sample['paragraphs'] if item['is_supporting']]
    gold_passages =  set([item['title'] + '\n' + item['paragraph_text'] for item in gold_passages])
    recall = dict()
    for k in k_list:
        recall[k] = sum(1 for t in gold_passages if t in retrieved_passages[:k]) / len(gold_passages)
        total[k] += recall[k]

for k in k_list:
    print(f"Recall@{k}: {total[k] / len(musique_data)}")

Recall@2: 0.34824999999999984
Recall@5: 0.46624999999999955
Recall@10: 0.5511666666666668
Recall@20: 0.6208333333333329
Recall@50: 0.6991666666666657
Recall@100: 0.7469166666666657


## 2WIKI

In [13]:
twowiki_contriever = Contriever()

with open("reproduce/dataset/2wikimultihopqa_corpus.json") as f:
    twowiki_corpus = json.load(f)
    twowiki_corpus = [item["title"] + '\n' + item["text"] for item in twowiki_corpus]

with open("reproduce/dataset/2wikimultihopqa.json") as f:
    twowiki_data = json.load(f)

all_queries = [sample['question'] for sample in twowiki_data]
len(all_queries), len(twowiki_corpus)

(1000, 6119)

In [14]:
twowiki_contriever.encode_corpus(twowiki_corpus)

Embedding Corpus: 48it [00:38,  1.24it/s]                        


In [15]:
top_results_twowiki = twowiki_contriever.batch_search(all_queries, top_k=100)

Embedding Query: 8it [00:00, 22.31it/s]                       


In [16]:
from collections import defaultdict

k_list = [ 2, 5, 10,20, 50, 100]
total = defaultdict(float)
for sample, retrieved_passages in zip(twowiki_data, top_results_twowiki):
    gold_passages = [item for item in sample['supporting_facts']]
    gold_items = set([item[0] for item in gold_passages])
    retrieved_items = [passage.split('\n')[0].strip() for passage in retrieved_passages]
    
    recall = dict()
    for k in k_list:
        recall[k] = sum(1 for t in gold_items if t in retrieved_items[:k]) / len(gold_items)
        total[k] += recall[k]

for k in k_list:
    print(f"Recall@{k}: {total[k] / len(musique_data)}")

Recall@2: 0.46575
Recall@5: 0.57525
Recall@10: 0.63475
Recall@20: 0.6845
Recall@50: 0.72375
Recall@100: 0.76


## HOTPOTQA

In [17]:
hotpotqa_contriever = Contriever()

with open("reproduce/dataset/hotpotqa_corpus.json") as f:
    hotpotqa_corpus = json.load(f)
    hotpotqa_corpus = [item["title"] + '\n' + item["text"] for item in hotpotqa_corpus]

with open("reproduce/dataset/hotpotqa.json") as f:
    hotpotqa_data = json.load(f)

all_queries = [sample['question'] for sample in hotpotqa_data]
len(all_queries), len(hotpotqa_corpus)

(1000, 9811)

In [18]:
# Encode the corpus
hotpotqa_contriever.encode_corpus(hotpotqa_corpus)

# Compute similarity scores for the queries
top_results_hotpotqa = hotpotqa_contriever.batch_search(all_queries, top_k=100)


Embedding Corpus: 77it [00:48,  1.58it/s]                        
Embedding Query: 8it [00:00, 10.40it/s]                       


In [19]:
from collections import defaultdict

k_list = [ 2, 5, 10,20, 50, 100]
total = defaultdict(float)
for sample, retrieved_passages in zip(hotpotqa_data, top_results_hotpotqa):
    gold_passages = [item for item in sample['supporting_facts']]
    gold_items = set([item[0] for item in gold_passages])
    retrieved_items = [passage.split('\n')[0].strip() for passage in retrieved_passages]
    
    recall = dict()
    for k in k_list:
        recall[k] = sum(1 for t in gold_items if t in retrieved_items[:k]) / len(gold_items)
        total[k] += recall[k]

for k in k_list:
    print(f"Recall@{k}: {total[k] / len(musique_data)}")

Recall@2: 0.584
Recall@5: 0.753
Recall@10: 0.834
Recall@20: 0.8825
Recall@50: 0.916
Recall@100: 0.936


## Formulate the loop

In [20]:
from collections import defaultdict


def encode_and_eval(all_corpus, all_queries, all_data, extract_function):
    contriever = Contriever()
    contriever.encode_corpus(all_corpus)
    top_results = contriever.batch_search(all_queries, top_k=100)
    k_list = [ 2, 5, 10,20, 50, 100]
    total = defaultdict(float)

    for sample, retrieved_passages in zip(all_data, top_results):
        gold_items, retrieved_items = extract_function(sample, retrieved_passages)
        recall = dict()
        for k in k_list:
            recall[k] = sum(1 for t in gold_items if t in retrieved_items[:k]) / len(gold_items)
            total[k] += recall[k]
    
    for k in k_list:
        print(f"Recall@{k}: {total[k] / len(all_data)}")

    return top_results, total

## popqa

In [22]:
with open("reproduce/dataset/popqa.json") as f:
    popqa_data = json.load(f)

all_queries = [sample['question'] for sample in popqa_data]

with open("reproduce/dataset/popqa_corpus.json") as f:
    popqa_corpus = json.load(f)
    popqa_corpus = [item["title"] + '\n' + item["text"] for item in popqa_corpus]

def extract_popqa(sample, retrieved_passages):
    gold_passages = [item for item in sample['paragraphs'] if item['is_supporting']]
    gold_items = set([item['title'] + '\n' + item['text'] for item in gold_passages])
    retrieved_items = retrieved_passages
    return gold_items, retrieved_items

len(popqa_data), len(popqa_corpus), len(all_queries)

(1000, 8676, 1000)

In [23]:
top_results_popqa, total_popqa = encode_and_eval(popqa_corpus, all_queries, popqa_data, extract_popqa)


Embedding Corpus: 68it [00:55,  1.23it/s]                        
Embedding Query: 8it [00:00,  9.40it/s]                       


Recall@2: 0.27
Recall@5: 0.432
Recall@10: 0.476
Recall@20: 0.491
Recall@50: 0.5015
Recall@100: 0.51


## NQ_REAR

In [24]:
with open("reproduce/dataset/nq_rear.json") as f:
    nq_rear_data = json.load(f)

all_queries = [sample['question'] for sample in nq_rear_data]

with open("reproduce/dataset/nq_rear_corpus.json") as f:
    nq_rear_corpus = json.load(f)
    nq_rear_corpus = [item["title"] + '\n' + item["text"] for item in nq_rear_corpus]

def extract_nq_rear(sample, retrieved_passages):
    gold_passages = [item for item in sample['contexts'] if item['is_supporting']]
    gold_items = set([item['title'] + '\n' + item['text'] for item in gold_passages])
    retrieved_items = retrieved_passages
    return gold_items, retrieved_items

len(nq_rear_data), len(nq_rear_corpus), len(all_queries)

(1000, 9633, 1000)

In [25]:
top_results_nq_rear, total_nq_rear = encode_and_eval(nq_rear_corpus, all_queries, nq_rear_data, extract_nq_rear)


Embedding Corpus: 76it [00:21,  3.58it/s]                        
Embedding Query: 8it [00:00, 29.31it/s]                       


Recall@2: 0.29118650793650747
Recall@5: 0.5456599206349206
Recall@10: 0.8343067460317459
Recall@20: 0.9313460317460317
Recall@50: 0.9728083333333333
Recall@100: 0.9873869047619048


# GTR

In [45]:
from typing import List, Union

import numpy as np
from sentence_transformers import SentenceTransformer


class GTR:
    def __init__(self, model_name='sentence-transformers/gtr-t5-base'):
        self.model = SentenceTransformer(model_name,device="cuda")
        self.batch_size = 128
        self.corpus: List[str] = []
        self.embedding: np.ndarray = None

    @torch.no_grad()
    def _embed(self, input_str):
        batch_embeddings = self.model.encode(input_str)
        norms = np.linalg.norm(batch_embeddings, axis=1, keepdims=True)
        batch_embeddings = batch_embeddings / norms
        return batch_embeddings

    def encode_corpus(self, corpus: List[str]):
        self.corpus = corpus
        embeddings = []
        for i in tqdm(range(0, len(corpus), self.batch_size), total=len(corpus)//self.batch_size, desc="Embedding Corpus"):
            embeddings.append(self._embed(corpus[i:i+self.batch_size]))
        self.embedding = np.concatenate(embeddings, axis=0)

    def batch_search(self, queries: Union[str, List[str]], top_k=5):
        if isinstance(queries, str):
            queries = [queries]

        query_embedding = []
        for i in tqdm(range(0, len(queries), self.batch_size), total=len(queries)//self.batch_size, desc="Embedding Query"):
            query_embedding.append(self._embed(queries[i:i+self.batch_size]))
        query_embedding = np.concatenate(query_embedding, axis=0)

        scores = np.matmul(query_embedding, self.embedding.T)

        top_k_idx = np.argsort(scores, axis=1)[:, -top_k:][:, ::-1]
        docs_to_return = []
        for top_k_idx_row in top_k_idx:
            top_k_docs = [self.corpus[i] for i in top_k_idx_row]
            docs_to_return.append(top_k_docs)
        if len(docs_to_return) == 1:
            docs_to_return = docs_to_return[0]
        return docs_to_return


In [46]:
from collections import defaultdict


def encode_and_eval_gtr(all_corpus, all_queries, all_data, extract_function):
    contriever = GTR("sentence-transformers/gtr-t5-base")
    contriever.encode_corpus(all_corpus)
    top_results = contriever.batch_search(all_queries, top_k=100)
    k_list = [ 2, 5, 10,20, 50, 100]
    total = defaultdict(float)

    for sample, retrieved_passages in zip(all_data, top_results):
        gold_items, retrieved_items = extract_function(sample, retrieved_passages)
        recall = dict()
        for k in k_list:
            recall[k] = sum(1 for t in gold_items if t in retrieved_items[:k]) / len(gold_items)
            total[k] += recall[k]
    
    for k in k_list:
        print(f"Recall@{k}: {total[k] / len(all_data)}")

    return top_results, total

### 2wiki

In [43]:
def extract_2wikimultihopqa(sample, retrieved_passages):
    gold_passages = [item for item in sample['supporting_facts']]
    gold_items = set([item[0] for item in gold_passages])
    retrieved_items = [passage.split('\n')[0].strip() for passage in retrieved_passages]
    return gold_items, retrieved_items

with open("reproduce/dataset/2wikimultihopqa.json") as f:
    twowiki_data = json.load(f)

all_queries = [sample['question'] for sample in twowiki_data]

with open("reproduce/dataset/2wikimultihopqa_corpus.json") as f:
    twowiki_corpus = json.load(f)
    twowiki_corpus = [item["title"] + '\n' + item["text"] for item in twowiki_corpus]

len(twowiki_data), len(twowiki_corpus), len(all_queries)

(1000, 6119, 1000)

In [47]:
top_results_twowiki_gtr, total_twowiki_gtr = encode_and_eval_gtr(twowiki_corpus, all_queries, twowiki_data, extract_2wikimultihopqa)

Embedding Corpus: 48it [00:20,  2.36it/s]                        
Embedding Query: 8it [00:00, 14.83it/s]                       


Recall@2: 0.60225
Recall@5: 0.67925
Recall@10: 0.711
Recall@20: 0.7335
Recall@50: 0.77075
Recall@100: 0.79825


## HOTPOT

In [48]:
def extract_hotpotqa(sample, retrieved_passages):
    gold_passages = [item for item in sample['supporting_facts']]
    gold_items = set([item[0] for item in gold_passages])
    retrieved_items = [passage.split('\n')[0].strip() for passage in retrieved_passages]
    return gold_items, retrieved_items

with open("reproduce/dataset/hotpotqa.json") as f:
    hotpotqa_data = json.load(f)
    all_queries = [sample['question'] for sample in hotpotqa_data]

with open("reproduce/dataset/hotpotqa_corpus.json") as f:
    hotpotqa_corpus = json.load(f)
    hotpotqa_corpus = [item["title"] + '\n' + item["text"] for item in hotpotqa_corpus]

len(hotpotqa_data), len(hotpotqa_corpus), len(all_queries)

(1000, 9811, 1000)

In [49]:
top_results_hotpotqa_gtr, total_hotpotqa_gtr = encode_and_eval_gtr(hotpotqa_corpus, all_queries, hotpotqa_data, extract_hotpotqa)

/home/qi.658/miniconda3/envs/hipporag/lib/python3.9/site-packages/sentence_transformers/models/Dense.py:77: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(os.path.

Recall@2: 0.5925
Recall@5: 0.739
Recall@10: 0.808
Recall@20: 0.8505
Recall@50: 0.9005
Recall@100: 0.923


## NQ

In [50]:
with open("reproduce/dataset/nq_rear.json") as f:
    nq_rear_data = json.load(f)

all_queries = [sample['question'] for sample in nq_rear_data]

with open("reproduce/dataset/nq_rear_corpus.json") as f:
    nq_rear_corpus = json.load(f)
    nq_rear_corpus = [item["title"] + '\n' + item["text"] for item in nq_rear_corpus]

def extract_nq_rear(sample, retrieved_passages):
    gold_passages = [item for item in sample['contexts'] if item['is_supporting']]
    gold_items = set([item['title'] + '\n' + item['text'] for item in gold_passages])
    retrieved_items = retrieved_passages
    return gold_items, retrieved_items

top_results_nq_rear_gtr, total_nq_rear_gtr = encode_and_eval_gtr(nq_rear_corpus, all_queries, nq_rear_data, extract_nq_rear)

/home/qi.658/miniconda3/envs/hipporag/lib/python3.9/site-packages/sentence_transformers/models/Dense.py:77: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(os.path.

Recall@2: 0.3497730158730151
Recall@5: 0.6339706349206348
Recall@10: 0.8798388888888881
Recall@20: 0.9463428571428569
Recall@50: 0.975122619047619
Recall@100: 0.9831638888888891


## POPQA

In [51]:
with open("reproduce/dataset/popqa.json") as f:
    popqa_data = json.load(f)

all_queries = [sample['question'] for sample in popqa_data]

with open("reproduce/dataset/popqa_corpus.json") as f:
    popqa_corpus = json.load(f)
    popqa_corpus = [item["title"] + '\n' + item["text"] for item in popqa_corpus]

def extract_popqa(sample, retrieved_passages):
    gold_passages = [item for item in sample['paragraphs'] if item['is_supporting']]
    gold_items = set([item['title'] + '\n' + item['text'] for item in gold_passages])
    retrieved_items = retrieved_passages
    return gold_items, retrieved_items

top_results_popqa_gtr, total_popqa_gtr = encode_and_eval_gtr(popqa_corpus, all_queries, popqa_data, extract_popqa)

/home/qi.658/miniconda3/envs/hipporag/lib/python3.9/site-packages/sentence_transformers/models/Dense.py:77: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(os.path.

Recall@2: 0.4005
Recall@5: 0.494
Recall@10: 0.5045
Recall@20: 0.51
Recall@50: 0.5185
Recall@100: 0.527
